# Product valuations: choose terms, then choose the question

The same market can value several contracts, but different contracts answer
different questions. A swap has a par fixed rate; a straight bond has a yield
and Z-spread; a caplet depends on a volatility surface as well as a forward rate.
This notebook prices all three and a cap strip, making those dependencies
explicit. See [setup](../README.md) and the [first swap](tour.ipynb) if needed.

In [1]:
import pandas as pd
import quantsupport as qs
from market_data import load_market

market = load_market()
ref = market.quotes.reference_date


## A swap: value the contract and solve its par coupon

The fixed rate is the term we vary. Everything else stays identical. Explicit
trade dates matter for forward-starting products: the default trade date on
many specifications is their start date, while we want valuation as of `ref`.

In [ ]:
def receive_fixed_swap(fixed_rate):
    instrument = qs.Swap(
        identifier="USD-SWAP-5Y",
        start_date=ref,
        maturity_date=ref + "5Y",
        notional=1_000_000,
        fixed_rate=fixed_rate,
        currency=qs.Currency.USD,
        market_index=qs.MarketIndex.SOFR,
        fixed_leg_frequency=qs.Frequency.Semiannual,
        floating_leg_frequency=qs.Frequency.Quarterly,
    )
    return qs.Trade(instrument, ref, qs.Side.LongReceive)


context = qs.PricingContext(
    quotes=market.quotes,
    curves=market.curves,
    csa_policy=market.csa_policy,
)

with context:
    swap_result = context.evaluate(
        receive_fixed_swap(0.04),
        [qs.Request.Value, qs.Request.FairRate],
    )
    par_result = context.evaluate(
        receive_fixed_swap(swap_result.fair_rate),
        [qs.Request.Value],
    )

pd.Series(
    {
        "original_pv_usd": swap_result.price,
        "par_rate": swap_result.fair_rate,
        "pv_at_par_usd": par_result.price,
    }
)

original_pv_usd   -3.479643e+03
par_rate           4.077000e-02
pv_at_par_usd      4.183676e-11
dtype: float64

## A straight bond: distinguish position value from a market price quote

We buy USD 1 million principal of a five-year, 4.5% fixed-rate bond. The context
selects the USD discount curve. A zero Z-spread first gives the curve-implied
value; a quoted clean price then asks which constant spread reconciles that
price with the discounted cashflows.

`BondPricingAnchor.clean_price(99.25)` means **99.25 per 100 principal**.
`result.price` remains the **USD position PV**. Yield-to-maturity and modified
duration use the bond's rate conventions; Z-spread is continuously compounded
and is added to the discount curve. These are different descriptions of the
same cashflows, not interchangeable input rates.

In [ ]:
bond = qs.FixedRateBond(
    identifier="USD-FIXED-BOND-5Y",
    start_date=ref,
    maturity_date=ref + "5Y",
    notional=1_000_000,
    rate=0.045,
    currency=qs.Currency.USD,
    day_counter=qs.DayCounter.Actual365,
    compounding=qs.Compounding.Compounded,
    rate_frequency=qs.Frequency.Semiannual,
    payment_frequency=qs.Frequency.Semiannual,
)
trade = qs.Trade(bond, ref, pricing_anchor=qs.BondPricingAnchor.z_spread(0.0))
context = qs.PricingContext(
    quotes=market.quotes,
    curves=market.curves,
    csa_policy=market.csa_policy,
)

with context:
    curve_value = context.evaluate(trade, [qs.Request.Value])
    bond_result = context.evaluate(
        trade.with_pricing_anchor(qs.BondPricingAnchor.clean_price(99.25)),
        [
            qs.Request.Value,
            qs.Request.YieldToMaturity,
            qs.Request.ModifiedDuration,
            qs.Request.ZSpread,
            qs.Request.Cashflows,
            qs.Request.Sensitivities,
        ],
    )
    spread_value = context.evaluate(
        trade.with_pricing_anchor(qs.BondPricingAnchor.z_spread(bond_result.z_spread)),
        [qs.Request.Value],
    )

pd.Series(
    {
        "curve_only_pv_usd": curve_value.price,
        "price_anchored_pv_usd": bond_result.price,
        "spread_anchored_pv_usd": spread_value.price,
        "anchor_pv_difference_usd": spread_value.price - bond_result.price,
        "ytm": bond_result.yield_to_maturity,
        "modified_duration_years": bond_result.modified_duration,
        "z_spread_bp": bond_result.z_spread * 10_000,
    }
)

curve_only_pv_usd          1.016330e+06
price_anchored_pv_usd      9.925000e+05
ytm                        4.669820e-02
modified_duration_years    4.429526e+00
z_spread_bp                5.155403e+01
dtype: float64

The spread is an independent risk factor after it has been solved. The
`<identifier>.ZSpread` row measures sensitivity to that spread; the other rows
measure curve-quote sensitivity **holding the solved spread fixed**. They do
not describe re-solving the spread to keep the observed clean price constant
under every market shock.

In [4]:
bond_result.sensitivities.assign(
    pnl_for_plus_1bp_usd=lambda frame: frame["value"] * 1e-4
)


,pillar,value,pnl_for_plus_1bp_usd
0,FixedRateDeposit_USD_SOFR_1D,-2.610029e+00,-0.000261
1,OIS_USD_SOFR_10Y,0.000000e+00,0.000000
2,OIS_USD_SOFR_1Y,-3.685470e+03,-0.368547
3,OIS_USD_SOFR_2Y,-7.121782e+03,-0.712178
4,OIS_USD_SOFR_30Y,0.000000e+00,0.000000
5,OIS_USD_SOFR_3Y,-1.526246e+04,-1.526246
6,OIS_USD_SOFR_5Y,-4.422541e+06,-442.254122
7,OIS_USD_SOFR_7Y,0.000000e+00,0.000000
8,USD-FIXED-BOND-5Y.ZSpread,-4.564299e+06,-456.429872


## A caplet and a strip of caplets

The caplet pays `notional × accrual × max(SOFR − strike, 0)` at the end of
one future accrual period. Fixing, accrual start, accrual end, and payment date
are separate terms even when some dates coincide. A `CapFloor` builds a strip
of these periods from a frequency.

The Python binding dispatches these products to the Black closed-form pricers.
The SOFR curve determines the forward and discount factor; the SOFR Black
surface provides volatility at the option expiry and strike. Attaching the
surface is necessary even though it was unnecessary for the swap and bond.

In [ ]:
fixing = ref + "1Y"
payment = fixing + "3M"
caplet = qs.CapletFloorlet(
    identifier="USD-CAPLET-1Y",
    fixing_date=fixing,
    start_accrual_date=fixing,
    end_accrual_date=payment,
    payment_date=payment,
    strike=0.045,
    currency=qs.Currency.USD,
    market_index=qs.MarketIndex.SOFR,
    payoff_type=qs.CapletFloorletType.Caplet,
)
cap = qs.CapFloor(
    identifier="USD-CAP-1Y-3Y",
    start_date=fixing,
    maturity_date=ref + "3Y",
    strike=0.045,
    notional=1_000_000,
    currency=qs.Currency.USD,
    market_index=qs.MarketIndex.SOFR,
    cap_floor_type=qs.CapFloorType.Cap,
    frequency=qs.Frequency.Quarterly,
)
context = qs.PricingContext(
    quotes=market.quotes,
    curves=market.curves,
    csa_policy=market.csa_policy,
    volatility_surfaces=market.volatility_surfaces,
)

with context:
    caplet_result = context.evaluate(
        qs.Trade(caplet, ref, notional=1_000_000),
        [qs.Request.Value, qs.Request.Sensitivities],
    )
    cap_result = context.evaluate(
        qs.Trade(cap, ref),
        [qs.Request.Value, qs.Request.Sensitivities],
    )

pd.DataFrame(
    [
        {"product": "single caplet", "pv_usd": caplet_result.price},
        {"product": "quarterly cap strip", "pv_usd": cap_result.price},
    ]
)

,product,pv_usd
0,single caplet,841.058728
1,quarterly cap strip,6328.853953


The caplet risk contains both interest-rate and volatility quote labels. A
volatility quote of `0.30` is a 30% Black volatility; a +1 volatility-point move
is `0.01`, whereas +1 bp on a rate is `0.0001`. Do not scale the whole table by
the same number and call it DV01. The next notebook separates these shocks.

In [6]:
caplet_result.sensitivities.sort_values(
    "value",
    key=lambda series: series.abs(),
    ascending=False,
).head(10)


,pillar,value
30,OIS_USD_SOFR_2Y,215199.303464
29,OIS_USD_SOFR_1Y,-110871.954680
7,CapletFloorlet_USD_SOFR_3M_1Y_Absolute_0.045_S...,3895.279899
27,FixedRateDeposit_USD_SOFR_1D,0.270744
25,CapletFloorlet_USD_SOFR_3M_7Y_Absolute_0.045_S...,0.000000
20,CapletFloorlet_USD_SOFR_3M_5Y_Absolute_0.055_S...,0.000000
21,CapletFloorlet_USD_SOFR_3M_6M_Absolute_0.035_S...,0.000000
22,CapletFloorlet_USD_SOFR_3M_6M_Absolute_0.045_S...,0.000000
23,CapletFloorlet_USD_SOFR_3M_6M_Absolute_0.055_S...,0.000000
24,CapletFloorlet_USD_SOFR_3M_7Y_Absolute_0.035_S...,0.000000


## Request guide for the products used here

| Contract | Useful requests in these examples | Interpretation |
| --- | --- | --- |
| Swap | `Value`, `FairRate`, `Cashflows`, `Sensitivities` | Position PV, par fixed coupon, payments, quote risk |
| Fixed-rate bond | `Value`, `YieldToMaturity`, `ModifiedDuration`, `ZSpread`, `Cashflows`, `Sensitivities` | Price/spread anchor and straight-bond analytics |
| Caplet / cap | `Value`, `Sensitivities` | Black option PV and curve/vol quote risk |
| Callable stepped bond | `Value`, `OAS`, `Sensitivities` | Model-dependent exercise value and spread risk |

This is a guide to supported combinations, not a request list to send to every
pricer. An unavailable result is not a zero. Callable debt deliberately has no
yield or modified-duration answer; see [stepped bonds](stepped_coupon_bond.ipynb).
Continue with [market changes](../volatilitysurface/curves_and_volatility.ipynb) or
[portfolio aggregation](../sensitivity/portfolio_risk.ipynb).